In [ ]:
import json, re, subprocess, sys, os
from pathlib import Path
from collections import defaultdict
import pandas as pd

def clone_repo(github_url, dest):
    dest_path = Path(dest)
    if dest_path.exists():
        print(f"{dest_path} already exists, skipping clone.")
        return dest_path
    result = subprocess.run(["git", "clone", "--depth", "1", github_url, str(dest_path)],
                             capture_output=True, text=True)
    if result.returncode != 0:
        print(f"CLONE FAILED for {github_url}:", result.stderr)
        return None
    print(f"Cloned {github_url} -> {dest_path}")
    return dest_path

def clone_repos(github_urls, base_dir="cloned_repos"):
    """Clones any number of repos, skipping ones that fail rather than
    aborting the whole batch -- one bad URL shouldn't block the others."""
    repo_dirs = []
    for i, url in enumerate(github_urls):
        repo_name = url.rstrip("/").split("/")[-1].replace(".git", "")
        dest = Path(base_dir) / f"{i}_{repo_name}"
        result = clone_repo(url, dest)
        if result:
            repo_dirs.append(result)
    print(f"\nSuccessfully cloned {len(repo_dirs)}/{len(github_urls)} repos")
    return repo_dirs

In [ ]:
def find_gradle_modules(repo_dir):
    files = list(repo_dir.rglob("build.gradle")) + list(repo_dir.rglob("build.gradle.kts"))
    return [f for f in files if "/.gradle/" not in str(f) and "/build/" not in str(f)]

DEP_LINE = re.compile(
    r"""(implementation|api|compileOnly|runtimeOnly|annotationProcessor|
         testImplementation|testCompile|testRuntime|compile|runtime)
        \s*\(?\s*['"](?P<coord>[^'"]+)['"]""", re.VERBOSE)

def parse_gradle_file(path):
    text = path.read_text()
    deps = []
    for m in DEP_LINE.finditer(text):
        parts = m.group("coord").split(":")
        if len(parts) >= 2:
            group, artifact = parts[0], parts[1]
            deps.append({"group": group, "artifact": artifact, "coordinate": f"{group}:{artifact}"})
    return deps

def collect_unique_artifacts(repo_dirs):
    """Aggregates declared dependencies across every cloned repo into one
    deduplicated list -- this is what feeds the LLM prompt, so each
    artifact only gets categorized once even if it appears in 50 services."""
    all_deps = {}  # artifact -> full coordinate (first one seen wins)
    per_repo_counts = defaultdict(int)

    for repo_dir in repo_dirs:
        modules = find_gradle_modules(repo_dir)
        for mod in modules:
            for dep in parse_gradle_file(mod):
                all_deps[dep["artifact"]] = dep["coordinate"]
                per_repo_counts[dep["artifact"]] += 1

    print(f"Found {len(all_deps)} unique artifacts across {len(repo_dirs)} repo(s)")
    return all_deps, per_repo_counts

In [ ]:
repo_urls = [
    "https://github.com/your-org/service-a.git",
    "https://github.com/your-org/service-b.git",
    "https://github.com/your-org/service-c.git",
]
repo_dirs = clone_repos(repo_urls)
unique_artifacts, per_repo_counts = collect_unique_artifacts(repo_dirs)

for artifact, coord in sorted(unique_artifacts.items()):
    print(f"  {coord}  (seen in {per_repo_counts[artifact]} build file(s))")

In [ ]:
PROMPT_TEMPLATE = """You are building a static reference playbook for a Java dependency-reduction
tool. I will give you a list of library dependencies found across our codebases.

DEPENDENCIES (artifact: full coordinate, number of repos it appears in):
{dep_list}

Task: group these by FUNCTIONAL CATEGORY (string-utils, json, datetime, http-client,
logging, encoding, testing, collections, etc). Only include a category in your
output if it has 2+ DIFFERENT libraries doing the same job -- skip categories
with only one library, and skip Spring Boot starters / framework dependencies
that don't have a real duplicate-functionality alternative in this list.

Respond ONLY with valid JSON, no prose, no markdown fences, in this exact shape:
{{
  "library_categories": [
    {{"artifact": "guava", "category": "string-utils"}}
  ],
  "categories": [
    {{"category": "string-utils", "artifacts": "guava,commons-lang3",
      "recommended": "commons-lang3",
      "reason": "one sentence, be specific and honest about real differences"}}
  ]
}}

Only include an artifact in library_categories if it belongs to a category with
2+ libraries (i.e. it appears in some "categories" entry). Do not guess wildly --
if you're not sure two libraries serve the same purpose, leave them out rather
than forcing a category match.
"""

def call_llm(prompt: str) -> dict:
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        return None
    import urllib.request
    body = json.dumps({
        "model": "claude-sonnet-4-6", "max_tokens": 2000,
        "messages": [{"role": "user", "content": prompt}],
    }).encode()
    req = urllib.request.Request(
        "https://api.anthropic.com/v1/messages", data=body,
        headers={"content-type": "application/json", "x-api-key": api_key,
                 "anthropic-version": "2023-06-01"},
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        data = json.loads(resp.read())
    text = "".join(b["text"] for b in data["content"] if b["type"] == "text")
    text = text.strip().removeprefix("```json").removesuffix("```").strip()
    return json.loads(text)

def batch_dicts(d, batch_size=25):
    items = list(d.items())
    for i in range(0, len(items), batch_size):
        yield dict(items[i:i+batch_size])

def categorize_artifacts(unique_artifacts: dict, per_repo_counts: dict, batch_size=25):
    """Batches large dependency lists (recommended: 20-30 per call) so the
    LLM's attention stays focused and output stays accurate."""
    all_results = {"library_categories": [], "categories": []}
    use_mock = "ANTHROPIC_API_KEY" not in os.environ
    if use_mock:
        print("No ANTHROPIC_API_KEY set -- using mock response for this test run.")

    for batch_num, batch in enumerate(batch_dicts(unique_artifacts, batch_size), 1):
        dep_list = "\n".join(f"  {art}: {coord} ({per_repo_counts[art]} repo(s))"
                              for art, coord in batch.items())
        prompt = PROMPT_TEMPLATE.format(dep_list=dep_list)
        print(f"\n--- Batch {batch_num} ({len(batch)} artifacts) ---")

        if use_mock:
            # Deterministic mock matching what a real call would plausibly
            # return for THIS exact batch's real artifacts, so the parsing/
            # Excel-writing steps downstream are tested against realistic data.
            result = mock_llm_response(batch)
        else:
            result = call_llm(prompt)

        if result:
            all_results["library_categories"].extend(result.get("library_categories", []))
            all_results["categories"].extend(result.get("categories", []))

    return all_results

def mock_llm_response(batch: dict) -> dict:
    """Mock fallback for sandbox testing only -- real runs with a key skip this."""
    lib_cats, cats = [], []
    if "guava" in batch:
        lib_cats.append({"artifact": "guava", "category": "string-utils"})
    if "jackson-databind" in batch:
        lib_cats.append({"artifact": "jackson-databind", "category": "json"})
    if "jackson-mapper-asl" in batch:
        lib_cats.append({"artifact": "jackson-mapper-asl", "category": "json"})
        lib_cats.append({"artifact": "jackson-core-asl", "category": "json"})
    if "joda-time" in batch:
        lib_cats.append({"artifact": "joda-time", "category": "datetime"})
    if any(a in batch for a in ["jackson-databind", "jackson-mapper-asl"]):
        cats.append({"category": "json", "artifacts": "jackson-databind,jackson-mapper-asl,jackson-core-asl",
                      "recommended": "jackson-databind",
                      "reason": "jackson-mapper-asl/jackson-core-asl are the legacy pre-2.0 Jackson, superseded by jackson-databind"})
    return {"library_categories": lib_cats, "categories": cats}

In [ ]:
def categorize_artifacts(unique_artifacts, per_repo_counts, batch_size=25):
    all_results = {"library_categories": [], "categories": []}  # accumulator, lives OUTSIDE the loop
    for batch_num, batch in enumerate(batch_dicts(unique_artifacts, batch_size), 1):
        result = call_llm(prompt)
        all_results["library_categories"].extend(result.get("library_categories", []))  # extend, not overwrite
        all_results["categories"].extend(result.get("categories", []))
    return all_results  # nothing written to disk yet

def generate_services_playbook(unique_artifacts, per_repo_counts, output_path="services_playbook.xlsx"):
    results = categorize_artifacts(unique_artifacts, per_repo_counts)  # loop runs and finishes FIRST
    lib_df = pd.DataFrame(results["library_categories"]).drop_duplicates(subset="artifact")
    cat_df = pd.DataFrame(results["categories"]).drop_duplicates(subset="category")
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:  # write happens ONCE, after both batches are in memory
        lib_df.to_excel(writer, sheet_name="library_categories", index=False)
        cat_df.to_excel(writer, sheet_name="categories", index=False)

In [ ]:
repo_urls = [
    "https://github.com/your-org/service-a.git",
    "https://github.com/your-org/service-b.git",
]
repo_dirs = clone_repos(repo_urls)
unique_artifacts, per_repo_counts = collect_unique_artifacts(repo_dirs)
results = categorize_artifacts(unique_artifacts, per_repo_counts, batch_size=25)

lib_cat_df = pd.DataFrame(results["library_categories"]).drop_duplicates(subset="artifact")
cat_df = pd.DataFrame(results["categories"]).drop_duplicates(subset="category")

with pd.ExcelWriter("playbook.xlsx", engine="openpyxl") as writer:
    lib_cat_df.to_excel(writer, sheet_name="library_categories", index=False)
    cat_df.to_excel(writer, sheet_name="categories", index=False)
print(f"Done: {len(lib_cat_df)} artifacts, {len(cat_df)} clusters -> playbook.xlsx")

In [ ]:
BRAINSTORM_PROMPT = """You are building a reference playbook of common Java/Spring Boot
dependency overlaps -- duplicate-functionality library pairs that show up
repeatedly across real-world codebases, independent of any specific repo.
Draw on general knowledge of the Java ecosystem.

Cover categories like: string-utils, json, datetime, http-client, logging,
collections, testing, dependency-injection, validation, caching, csv/excel
parsing, PDF generation, mocking frameworks, etc.

IMPORTANT CAVEAT: your knowledge has a training cutoff and may be stale on
"latest"/"currently maintained" claims. Where unsure, say so explicitly in
the reason field rather than asserting confidently.

Respond ONLY with valid JSON, same shape as the repo-scan output:
{
  "library_categories": [{"artifact": "...", "category": "..."}],
  "categories": [{"category": "...", "artifacts": "a,b", "recommended": "...", "reason": "..."}]
}
Only include genuinely well-known overlaps you're confident about.
"""

def generate_brainstorm_playbook(output_path="brainstorm_playbook.xlsx"):
    if "ANTHROPIC_API_KEY" not in os.environ:  # adjust for your Vertex AI check
        print("No LLM credentials set -- skipping.")
        return None, None

    result = call_llm(BRAINSTORM_PROMPT)

    lib_df = pd.DataFrame(result.get("library_categories", [])).drop_duplicates(subset="artifact")
    cat_df = pd.DataFrame(result.get("categories", [])).drop_duplicates(subset="category")

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        lib_df.to_excel(writer, sheet_name="library_categories", index=False)
        cat_df.to_excel(writer, sheet_name="categories", index=False)

    print(f"brainstorm_playbook: {len(lib_df)} artifacts, {len(cat_df)} clusters -> {output_path}")
    return lib_df, cat_df

In [ ]:
repo_urls = ["https://github.com/your-org/service-a.git", "https://github.com/your-org/service-b.git"]
repo_dirs = clone_repos(repo_urls)
unique_artifacts, per_repo_counts = collect_unique_artifacts(repo_dirs)

services_lib_df, services_cat_df = generate_services_playbook(unique_artifacts, per_repo_counts)
brainstorm_lib_df, brainstorm_cat_df = generate_brainstorm_playbook()

In [ ]:
SIMPLE_CATEGORIZE_PROMPT = """Classify each Java/Spring library below into a short,
lowercase, hyphenated functional category (e.g. string-utils, json, datetime,
http-client, logging, encoding, testing, messaging, collections). Use an
existing well-known category name where it clearly applies; only invent a
new one if nothing fits. Do NOT try to detect duplicates or pair libraries --
just assign one category label per artifact, independently.

LIBRARIES:
{artifact_list}

Respond ONLY with valid JSON: {{"artifact_name": "category_name", ...}}
"""

def categorize_artifacts_pass1(unique_artifacts: dict, batch_size=25) -> dict:
    """PASS 1: collect artifact -> category from every batch. Each call only
    sees its own batch, but that's fine now -- this step never needs
    cross-batch context, it's just independent per-artifact tagging."""
    all_artifact_categories = {}
    for batch_num, batch in enumerate(batch_dicts(unique_artifacts, batch_size), 1):
        artifact_list = "\n".join(batch.keys())
        prompt = SIMPLE_CATEGORIZE_PROMPT.format(artifact_list=artifact_list)
        print(f"--- Batch {batch_num} ({len(batch)} artifacts) ---")
        result = call_llm(prompt)  # returns {"artifact": "category", ...}
        if result:
            # normalize to guard against spelling/casing drift between calls
            normalized = {k.strip().lower(): v.strip().lower() for k, v in result.items()}
            all_artifact_categories.update(normalized)
    return all_artifact_categories


def cluster_from_categories(all_artifact_categories: dict) -> dict:
    """PASS 2: clustering happens HERE, locally, AFTER every batch has been
    collected. This is correct by construction -- it's plain Python dict
    grouping, not an LLM judgment call, so it cannot miss a cross-batch
    pairing no matter how artifacts got split across calls."""
    clusters = defaultdict(list)
    for artifact, category in all_artifact_categories.items():
        clusters[category].append(artifact)
    return {cat: artifacts for cat, artifacts in clusters.items() if len(artifacts) > 1}

In [ ]:
artifact_categories = categorize_artifacts_pass1(unique_artifacts, batch_size=25)
duplicate_clusters = cluster_from_categories(artifact_categories)
print(duplicate_clusters)

In [ ]:
def get_recommendations_for_clusters(duplicate_clusters: dict) -> list:
    cluster_text = "\n".join(f"{cat}: {', '.join(artifacts)}" for cat, artifacts in duplicate_clusters.items())
    prompt = f"""For each of these CONFIRMED duplicate-functionality clusters, recommend
which library to standardize on and give a one-sentence reason:

{cluster_text}

Respond ONLY as JSON: [{{"category": "...", "recommended": "...", "reason": "..."}}]
"""
    return call_llm(prompt) or []

In [ ]:
# Handles BOTH declaration styles found in real Gradle Groovy DSL:
#
# String style (single quoted coordinate):
#   implementation 'group:artifact:version'
#   annotationProcessor 'group:artifact'       <- BOM-managed, no version
#
# Map style (named arguments):
#   implementation group: 'x', name: 'y', version: 'z'
#   annotationProcessor group: 'x', name: 'y'  <- version optional
#
DEP_LINE = re.compile(
    r"""(implementation|api|compileOnly|runtimeOnly|annotationProcessor|
         testImplementation|testCompile|testRuntime|compile|runtime)
        \s*\(?
        (?:
          ['"](?P<coord>[^'"]+)['"]
          |
          group\s*:\s*['"](?P<group>[^'"]+)['"]
          (?:\s*,\s*name\s*:\s*['"](?P<name>[^'"]+)['"])?
          (?:\s*,\s*version\s*:\s*['"](?P<version>[^'"]+)['"])?
        )""", re.VERBOSE)

def parse_gradle_file(path: Path) -> list:
    text = path.read_text()
    deps = []
    for m in DEP_LINE.finditer(text):
        scope = m.group(1)
        if m.group("coord"):
            # string style: 'group:artifact:version'
            parts = m.group("coord").split(":")
            if len(parts) < 2:
                continue
            group, artifact = parts[0], parts[1]
            version = parts[2] if len(parts) > 2 else "MANAGED"
        elif m.group("group"):
            # map style: group: '...', name: '...', version: '...'
            group = m.group("group")
            artifact = m.group("name") or "UNKNOWN"
            version = m.group("version") or "MANAGED"
        else:
            continue
        deps.append({
            "scope": scope,
            "group": group,
            "artifact": artifact,
            "version": version,
            "coordinate": f"{group}:{artifact}",
        })
    return deps